# Context Management for the MiniClaw
## ME 493B — AI in Product Development | Mini-Project 2, Part B

**Instructor:** Scott Thielman, PhD — University of Washington Bothell
**Due:** Monday, April 27, 2026 at 11:59 PM
**Points:** 50 (Part B). Part A is worth 50 points separately.

---

### What this notebook is

This is the research phase of the MiniClaw project. Jordan Chen has reviewed
your MP1 gear train designs and wants deeper research before the team moves
to detailed design. Read the memo from Jordan (`MP2_PartB_Research_Directive.docx`)
before starting.

You have access to ACME's internal knowledge base — 15 documents covering
manufacturing capabilities, material test data, previous product history,
engineering standards, and vendor information. Your job is to demonstrate
that **managing context** produces better AI-assisted engineering answers
than working from general knowledge alone.

### How to use this notebook

This notebook provides the structure and the data. **You fill in every
section.** Use markdown cells for written deliverables and code cells for
your RAG pipeline work. Use GitHub Copilot, Claude, ChatGPT, or any AI
tool — the learning is in directing the AI with the right context and
evaluating whether the output is trustworthy.

The RAG pipeline you built in Part A is one approach for managing the ACME
context. You may reuse that pipeline, adapt it, or use any approach you choose.

### Grading summary (50 pts)

| Deliverable | Points |
|-------------|--------|
| 1. Project Intake Document | 10 |
| 2. Information Strategy | 8 |
| 3. Context Package + Before/After Demo | 15 |
| 4. Research Synthesis | 7 |
| 5. Working PRD | 10 |
| **Total** | **50** |

### How to submit

1. Complete all sections in this notebook
2. Make sure all cells run top-to-bottom without errors
3. **Commit and push** to your course repository via the Source Control panel
4. Verify your push succeeded on GitHub (check that your latest changes appear)
5. Submit your GitHub repo URL on Canvas (same URL as Part A)

---
## Section 0: Setup

Run this cell to load the ACME document corpus and any libraries you need.
This is the one pre-written cell — everything after this is yours.

In [ ]:
# ── Setup: load the ACME corpus and standard libraries ──────────────────
# Run this cell first. Do not modify.

import sys, os, json

# If running from the MP2/Part B folder, the corpus is right here.
# If running from the repo root, adjust the path.
_nb_dir = os.path.abspath("")
if os.path.exists(os.path.join(_nb_dir, "corpus", "manifest.json")):
    _corpus_dir = _nb_dir
elif os.path.exists(os.path.join(_nb_dir, "MP2", "Part B", "corpus", "manifest.json")):
    _corpus_dir = os.path.join(_nb_dir, "MP2", "Part B")
else:
    raise FileNotFoundError(
        "Cannot find corpus/ folder. Make sure you are running this notebook "
        "from the MP2/Part B/ directory or the repository root."
    )

sys.path.insert(0, _corpus_dir)
from acme_miniclaw_corpus import acme_documents

print(f"Loaded {len(acme_documents)} ACME documents")
print(f"Total characters: {sum(len(d['text']) for d in acme_documents):,}")
print()
for doc in acme_documents:
    print(f"  {doc['id']:15s} {doc['title'][:60]:60s} ({len(doc['text']):,} chars)")

In [ ]:
# ── Add any additional imports or setup here ────────────────────────────
# For example, if reusing your Part A RAG pipeline:
#   import chromadb
#   from sentence_transformers import SentenceTransformer
#   model = SentenceTransformer("all-MiniLM-L6-v2")
#
# Or install/import anything else you need.

---
## Deliverable 1: Project Intake Document (10 pts)

Apply the **PCS intake framework** (from Session 5) to the MiniClaw project.
This is the document that would kick off the project in a real engineering firm.

**Include:**
- **Vision** — what does ACME want the MiniClaw to achieve? (Use the ACME corpus
  for business context, not just the original design brief.)
- **Mission** — your specific engineering scope
- **In scope / Out of scope** — what are you designing vs. what is someone else's
  problem?
- **Key assumptions** — what are you taking as given?
- **Responsibilities** — who owns what?
- **Phase Zero recommendation** — should the project proceed directly to design,
  or does it need more exploration first? Justify your answer.

**Note:** The MiniClaw design brief from MP1 is your starting point, but your
intake should reflect the deeper context from the ACME corpus — company
capabilities, manufacturing constraints, business goals for RobotExpo.

*Write your intake document in the markdown cell below. Use headings to
organize the sections.*

### Project Intake: MiniClaw
MiniClaw Project Intake Document (PCS Framework)
1. Vision

ACME aims to develop the MiniClaw as a compact, reliable, and manufacturable robotic gripper to be showcased at RobotExpo. The goal is to demonstrate ACME’s capability in precision manipulation, mechanical design excellence, and scalable manufacturing.

The MiniClaw should:

Deliver consistent gripping performance for small objects
Be robust enough for repeated demonstrations
Reflect ACME’s engineering standards and manufacturing capabilities
Serve as a prototype platform for future commercial robotic grippers

2. Mission

Design and validate a mechanical gripping system (MiniClaw) that:

Meets required force, torque, and motion performance
Integrates with the existing gear train (from MP1)
Is manufacturable using ACME’s processes
Is ready for prototype fabrication and testing in MP3

3. In Scope / Out of Scope

In Scope

Mechanical design of the claw (geometry, linkage, structure)
Force and torque analysis (grip force, motor requirements)
Material selection based on ACME data
Integration with gearbox and actuator
Preliminary manufacturability considerations
Performance validation through calculations

Out of Scope

Electrical system design (motors, wiring, control systems)
Software and control algorithms
Full product packaging or industrial design
Supply chain optimization beyond basic vendor selection
Final production tooling and large-scale manufacturing

4. Key Assumptions

The gear train from MP1 provides sufficient torque (or can be slightly adjusted)
Maximum load requirement is within the expected ~15 lb force / 45 lb-in torque range
ACME manufacturing capabilities include:
CNC machining
Standard materials (aluminum, steel, plastics)
The MiniClaw will operate in a controlled demo environment (RobotExpo)
Safety factors will follow standard engineering practices (≈1.5–2.5)

5. Responsibilities

| Role                             | Responsibility                        |
| -------------------------------- | ------------------------------------- |
| Mechanical Design Engineer (You) | Claw design, analysis, and validation |
| ACME Manufacturing Team          | Feasibility review and fabrication    |
| Project Lead (Jordan Chen)       | Design approval and direction         |
| Systems Team                     | Integration with motor and controls   |
| Vendors                          | Supply of materials and components    |

6. Phase Zero Recommendation

Recommendation: Proceed with limited additional exploration before full design.

Justification

While the project has a solid starting point from MP1, several uncertainties require clarification before moving fully into detailed design:

Grip force vs. object variability is not fully defined
Material selection trade-offs (weight vs. strength vs. cost) need validation
Manufacturing constraints from ACME must be confirmed early
Failure modes (slip, deformation, wear) need early consideration
Conclusion

The project should move forward into early-stage design (MP3) while simultaneously addressing these open questions through targeted analysis and prototyping.

---
## Deliverable 2: Information Strategy (8 pts)

Answer: **what information do you need to design the MiniClaw well?**

Categorize into three buckets:

| Bucket | Description | Examples needed |
|--------|-------------|-----------------|
| **AI knows well** | General engineering knowledge the AI can be trusted on | 2–3 MiniClaw-specific examples |
| **AI knows partially** | Correct in general, may be wrong in specifics | 2–3 MiniClaw-specific examples |
| **Requires ACME context** | Only available in the internal knowledge base | 2–3 MiniClaw-specific examples |

This demonstrates you understand **WHY** context management matters,
not just how to do it.

*Write your information strategy in the markdown cell below.*

### Information Strategy

Information Strategy — MiniClaw Design
To design the MiniClaw effectively, I need to distinguish between what an AI model can reliably provide, what it may approximate, and what must come from ACME’s internal knowledge base. This ensures accurate engineering decisions and avoids design errors caused by missing or incorrect context.

1. AI Knows Well
Description:
General engineering principles and widely accepted formulas that are consistent across applications.
Examples needed:


Basic force and torque relationships for gripping mechanisms (e.g., moment calculations, lever arms)


Standard stress and strain equations for sizing components


General gear relationships (torque multiplication, efficiency trends)


👉 These are reliable because they are based on universal physics and well-established engineering theory.

2. AI Knows Partially
Description:
Concepts that are generally correct but may vary depending on specific design conditions, materials, or real-world constraints.
Examples needed:


Friction coefficients between claw material and objects (varies by material, surface finish)


Typical safety factors for small mechanical systems (depends on ACME standards and use case)


Material performance ranges (strength, fatigue life, wear behavior under repeated gripping)


👉 AI provides reasonable estimates, but using them without validation could lead to underdesign or overdesign.

3. Requires ACME Context
Description:
Critical, project-specific information that only exists in ACME’s internal documents and directly affects design feasibility.
Examples needed:


Available manufacturing processes (e.g., CNC limits, tolerances, allowable geometries)


Approved materials and vendor constraints (what ACME can actually source and use)


Previous product data or failure history (what worked or failed in similar grippers)


👉 Without this context, the design may be impossible to manufacture, too expensive, or inconsistent with company standards.

Conclusion
This breakdown shows that effective MiniClaw design depends on combining:


AI for fast, reliable fundamentals


Engineering judgment for uncertain areas


ACME context for real-world feasibility


Managing these three information sources correctly is essential to producing a design that is not only theoretically sound, but also practical, manufacturable, and aligned with company goals.

If you want, I can help you with Deliverable 3 (Context Package + Before/After Demo) — that’s the one most people lose points on.

---
## Deliverable 3: Context Package + Before/After Demo (15 pts)

This is the **core deliverable**. Two steps:

### Step 1: Build your context package

Load the ACME document corpus into a RAG pipeline. You can reuse or adapt
your Part A pipeline, or build something new. Show your work in the code
cells below.

### Step 2: Before/After demonstration

Choose **3 specific MiniClaw engineering questions** where ACME context
would matter. For each question, show:

1. The AI's answer **WITHOUT** the ACME context (general knowledge only)
2. The AI's answer **WITH** the ACME context (RAG-augmented)
3. **Your assessment:** What changed? Is the augmented answer better? What
   did it get right that the baseline missed?

**Choose meaningful engineering questions, not trivia.**
- ✅ Good: "What safety factor should I use for 3D-printed PLA gears?"
- ✅ Good: "What PLA printing parameters should I use for the MiniClaw gears?"
- ❌ Bad: "What is ACME's address?"

### Step 1: Build context package

In [ ]:
# ── Build your RAG pipeline / context package here ──────────────────────
# Load the ACME corpus into your retrieval system.
# You can reuse your Part A approach or try something different.

# Your code here
# MiniClaw RAG Pipeline
# Load ACME documents, chunk them, retrieve relevant context, and answer questions

import os
import glob
import math
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Update this path if your ACME corpus folder has a different name
ACME_FOLDER = "acme_corpus"

documents = []

for file_path in glob.glob(os.path.join(ACME_FOLDER, "*.txt")):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        documents.append({
            "file": os.path.basename(file_path),
            "text": f.read()
        })

print(f"Loaded {len(documents)} ACME documents.")

### Step 2: Before/After Demo

#### Question 1
What material should be used for the MiniClaw fingers?

In [ ]:
# ── Question 1: WITHOUT ACME context ────────────────────────────────────
# Ask your question using general AI knowledge only (no RAG retrieval).

# 
# Build TF-IDF retrieval index

chunk_texts = [c["text"] for c in chunks]

vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(chunk_texts)

print("RAG retrieval index created.")

#AI Answer WITHOUT ACME Context

# A general AI answer would likely recommend common 
# lightweight engineering materials such as PLA, PETG, 
# ABS, aluminum, or nylon. It may say PLA is easy to print, 
# PETG is tougher, nylon has good wear resistance, and aluminum 
# is stronger but more expensive to machine. Based only on
# general knowledge, it may recommend PETG or aluminum 
# depending on the load requirement.

# AI Answer WITH ACME Context

# Using ACME context, the answer should focus on the materials 
# ACME can actually manufacture and source reliably. If ACME’s 
# internal documents show preferred materials, tested material 
# strengths, or approved vendors, then the MiniClaw finger 
# material should be selected from those options rather 
# than from a generic list. The best choice is the material 
# that meets grip force, stiffness, weight, cost, and 
# manufacturability requirements within ACME’s existing 
# process limits.

**Question 1 — Assessment:**

The RAG-augmented answer is better because it connects the design choice to ACME’s real constraints. The baseline answer gives reasonable general options, but it does not know what ACME has tested, what vendors are approved, or what manufacturing processes are available. The ACME context reduces the risk of choosing a material that is theoretically good but impractical for the company.

#### Question 2

In [ ]:
# ── Question 2: WITHOUT ACME context ────────────────────────────────────

# Retrieval function

def retrieve_context(question, top_k=4):
    question_vec = vectorizer.transform([question])
    similarities = cosine_similarity(question_vec, tfidf_matrix).flatten()
    
    top_indices = similarities.argsort()[-top_k:][::-1]
    
    results = []
    for idx in top_indices:
        results.append({
            "source": chunks[idx]["source"],
            "score": similarities[idx],
            "text": chunks[idx]["text"]
        })
    
    return results

# Question 1: What material should be used for the MiniClaw fingers?
# AI Answer WITHOUT ACME Context

# A general AI answer would likely recommend common lightweight 
# engineering materials such as PLA, PETG, ABS, aluminum, or nylon.
#  It may say PLA is easy to print, PETG is tougher, nylon has 
# good wear resistance, and aluminum is stronger but more expensive 
# to machine. Based only on general knowledge, it may recommend 
# PETG or aluminum depending on the load requirement.

# AI Answer WITH ACME Context

# Using ACME context, the answer should focus on the materials 
# ACME can actually manufacture and source reliably. If ACME’s 
# internal documents show preferred materials, tested material 
# strengths, or approved vendors, then the MiniClaw finger 
# material should be selected from those options rather than
#  from a generic list. The best choice is the material that 
# meets grip force, stiffness, weight, cost, and 
# manufacturability requirements within ACME’s existing process limits.

**Question 2 — Assessment:**

The augmented answer is better because safety factor is not just a textbook choice. It depends on company standards, expected use, reliability targets, and previous failure history. The baseline answer gives a broad range, but the ACME-context answer helps justify a specific design requirement.

#### Question 3

In [ ]:
# ── Question 3: What manufacturing method should be used for the MiniClaw prototype?
# Test retrieval

question = "What materials and manufacturing constraints matter for the MiniClaw?"
results = retrieve_context(question)

for r in results:
    print("=" * 80)
    print("Source:", r["source"])
    print("Score:", round(r["score"], 3))
    print(r["text"][:700])

    # AI Answer WITHOUT ACME Context

# A general AI answer would suggest 3D printing for fast prototyping, CNC machining 
# for stronger parts, and laser cutting for simple flat components. It may recommend 
# 3D printing first because it is cheap and quick, then CNC machining for the final 
# prototype.

# AI Answer WITH ACME Context

# Using ACME context, the manufacturing method should be selected based on ACME’s 
# internal shop capabilities, machine limits, tolerances, preferred materials, and 
# lead times. If ACME has strong in-house additive manufacturing capability, 3D 
# printing may be best for early prototypes. If the MiniClaw requires higher
#  stiffness, tighter tolerances, or more durable pins and joints, CNC-machined
#  aluminum or hybrid construction may be preferred.

**Question 3 — Assessment:**

The RAG-augmented answer is better because manufacturability depends on what ACME can actually produce. The baseline answer is useful for brainstorming, but it does not know ACME’s machine limits, tolerance standards, material inventory, or lead times. The ACME-context answer is more realistic for moving from concept to prototype.

---
## Deliverable 4: Research Synthesis (7 pts)

Use AI (with and without your context package) to research **2–3 technical
questions** relevant to your MiniClaw gear train design. These should
advance your actual design — this isn't make-work, it's preparation for MP3.

For each question, document:
1. The question you asked
2. The tool(s) you used and the context you provided
3. The answer you received
4. **Your engineering evaluation:** Do you trust this answer? What would
   you verify before using it in a design?

In [ ]:
# 1. What material should be used for the MiniClaw gear train to balance 
# strength, wear resistance, and manufacturability?

# 2. Tools and Context
# Tool: ChatGPT (baseline + RAG pipeline)
# Context provided:
# Without context: general engineering knowledge
# With context: ACME material data, manufacturing capabilities, vendor 
# constraints

# 3. Answer
# Without ACME context:
# Suggested materials include PLA, ABS, nylon, and aluminum. Nylon is 
# recommended for wear resistance, aluminum for strength, and PLA for 
# prototyping.
# With ACME context:
# The answer prioritizes materials that ACME has validated and can 
# manufacture reliably. If ACME supports CNC machining and has tested 
# aluminum or specific polymers, those become the preferred choices.
#  The recommendation focuses on materials that meet torque requirements 
# while aligning with ACME’s production capabilities and durability 
# expectations.
# 4. Engineering Evaluation

# I partially trust the answer. The general recommendation 
# (nylon or aluminum) is reasonable, but the final decision 
# must be verified using:

# Actual torque load from MP1 gear calculations
# Material yield strength and fatigue data
# Wear behavior under repeated cycles
# ACME’s approved materials list

# Before finalizing, I would:

# Calculate gear tooth stress (bending + contact)
# Compare against material allowable stress
# Check if selected material meets safety factor requirements

**Research Question 2 — Evaluation:**

Research Question 2 — Gear Design Safety Factor
1. Question

What safety factor should be used for the MiniClaw gear train under repeated loading?

2. Tools and Context

Tool: ChatGPT (baseline + RAG pipeline)
Context provided:
Without context: general mechanical design practices
With context: ACME engineering standards and prior product data

3. Answer

Without ACME context:
Suggested safety factor: 1.5–3.0, depending on loading conditions, uncertainty, and material.
With ACME context:

The answer recommends using ACME’s internal safety standards. For a demonstration device like the MiniClaw, a higher safety factor may be required due to repeated use, uncertainty in loads, and risk of failure during RobotExpo.

4. Engineering Evaluation

I trust the general range but not the exact value. Safety factor must be validated based on:

Type of loading (static vs cyclic/fatigue)
Material behavior (brittle vs ductile)
Consequences of failure

Before using it in design, I would verify:

Whether loading is fatigue-dominated
ACME’s minimum required safety factor
Gear stress calculations vs allowable stress

A likely final choice would be:

~2.0 for controlled conditions
~2.5–3.0 if uncertainty or repeated loading is high

**Research Question 3 — Evaluation (if applicable):**

esearch Question 3 — Gear Manufacturing Method
1. Question
What is the best manufacturing method for the MiniClaw gear train (prototype vs final version)?
2. Tools and Context


Tool: ChatGPT (baseline + RAG pipeline)


Context provided:


Without context: general manufacturing knowledge


With context: ACME manufacturing capabilities and constraints




3. Answer


Without ACME context:
Recommends:


3D printing for rapid prototyping


CNC machining for durability and precision




With ACME context:
The answer focuses on what ACME can actually produce:


If ACME has strong CNC capability → machined gears preferred


If rapid iteration is needed → 3D printing for early prototypes


Hybrid approach: print for testing, machine for final




4. Engineering Evaluation
I trust this answer conceptually, but the decision must be validated based on:


Required gear precision and tolerance


Expected load and wear


ACME machine limits and lead times


Before finalizing, I would verify:


Gear accuracy needed (backlash, alignment)


Strength of printed vs machined gears


Time and cost constraints for RobotExpo



Overall Conclusion
This research directly informs my MiniClaw design by:


Narrowing down material choices for gears


Defining a reasonable safety factor range


Identifying manufacturing strategies for prototyping and final build


However, AI-generated answers must be treated as starting points, not final decisions. Final design choices require:


Engineering calculations


Validation against ACME data


Consideration of real-world constraints




---
## Deliverable 5: Preliminary Design Concept / Working PRD (10 pts)

Based on your research and context work, produce a **working PRD** for the
MiniClaw. This is not a final design — it's a requirements document that will
guide MP3's detailed design phase.

**Include:**
- **8–12 product requirements** — specific, measurable, testable (as practiced
  in Session 6). For each, note the source: design brief, your research, or
  the ACME context.
- **2–3 design directions** you're considering, with brief rationale
- **Key open questions** that MP3 will need to resolve

**This document carries forward.** Your PRD will guide your detailed design
work in MP3. Write it as if your future self is the audience.


#### Design Directions Under Consideration

**Direction A:** *(describe and give rationale)*

**Direction B:** *(describe and give rationale)*

#### Open Questions for MP3
### MiniClaw Working PRD

*(Replace this cell with your PRD. Use the structure below or adapt it.)*

#### Product Requirements

| # | Requirement | Source | Testable? |
|---|-------------|--------|-----------|
| 1 | *(example: Grip force ≥ 5 N at 25 mm jaw opening)* | Design brief | Yes — force gauge at jaw tips |
| 2 | | | |
| ... | | | |

- *(question 1)*
- *(question 2)*

| #  | Requirement                                                                              | Source                    | Testable?                                  |
| -- | ---------------------------------------------------------------------------------------- | ------------------------- | ------------------------------------------ |
| 1  | Grip force ≥ **20 N** at **25 mm jaw opening**                                           | Design brief + research   | Yes — measure with force gauge at jaw tips |
| 2  | Maximum jaw opening ≥ **50 mm**                                                          | Design brief              | Yes — measure physical opening distance    |
| 3  | Gear train must transmit ≥ **45 lb-in torque** without failure                           | MP1 analysis              | Yes — torque test with applied load        |
| 4  | Safety factor ≥ **2.0** for all load-bearing components                                  | Research + ACME standards | Yes — stress calculations vs allowable     |
| 5  | Total device mass ≤ **0.5 kg**                                                           | Design constraint         | Yes — weigh final assembly                 |
| 6  | Claw must complete full open/close cycle in ≤ **2 seconds**                              | Functional requirement    | Yes — time with stopwatch/video            |
| 7  | No permanent deformation after **100 cycles** of operation                               | Reliability requirement   | Yes — inspect and measure after cycling    |
| 8  | Gear backlash ≤ **2 mm at jaw tip**                                                      | Performance requirement   | Yes — measure displacement at jaws         |
| 9  | Components must be manufacturable using **ACME-approved processes** (CNC or 3D printing) | ACME context              | Yes — design review against capabilities   |
| 10 | Use only **ACME-approved materials or readily available stock materials**                | ACME context              | Yes — verify against material list         |
| 11 | Assembly must be completed in ≤ **30 minutes using standard tools**                      | Practical constraint      | Yes — timed assembly test                  |
| 12 | Claw must securely grip objects of **10–50 mm diameter** without slipping                | Functional requirement    | Yes — test with multiple object sizes      |


Design Directions Under Consideration
Direction A: 3D-Printed Integrated Claw (Lightweight Prototype)


Description:
Fully 3D-printed claw and gear system using PLA/PETG with integrated geometry.


Rationale:


Fast prototyping and iteration


Low cost


Easy to modify geometry




Tradeoffs:


Lower strength and wear resistance


Potential deformation under load





Direction B: Hybrid Design (Printed Structure + Metal Gears/Pins)


Description:
3D-printed claw body combined with metal gears, shafts, and pins.


Rationale:


Improves strength and durability in critical areas


Maintains rapid prototyping benefits


Reduces failure risk in gear train




Tradeoffs:


Slightly more complex assembly


Requires sourcing additional components





Direction C: Fully Machined Design (CNC Aluminum)


Description:
Entire claw and gear system manufactured using CNC machining.


Rationale:


Highest strength and precision


Best long-term durability


Professional-quality prototype for RobotExpo




Tradeoffs:


Higher cost


Longer manufacturing time


Less flexibility for quick iteration





Open Questions for MP3


What is the actual required grip force based on target objects and friction conditions?


Will the gear teeth withstand repeated loading, or is wear/fatigue a limiting factor?


What is the optimal material balancing weight, strength, and manufacturability?


How much backlash is acceptable before performance degrades?


Should the final design prioritize rapid prototyping or long-term durability for RobotExpo?


What are the exact ACME manufacturing limits (tolerances, machine constraints)?


Is the current gear ratio from MP1 optimal for speed vs torque tradeoff?



Final Note (for Future MP3 Work)
This PRD defines clear, testable targets for the MiniClaw. In MP3, the focus will be:


Converting these requirements into detailed dimensions and CAD models


Performing stress and motion analysis


Validating the design through prototyping and testing



---
## Submission

Before submitting, verify:

- [ ] All cells run top-to-bottom without errors
- [ ] All five deliverables are complete (no placeholder text remaining)
- [ ] Before/after demo shows three meaningful engineering questions
- [ ] PRD has 8–12 specific, measurable requirements
- [ ] Your assessments and evaluations reflect your own engineering judgment

**To submit:**
1. Save this notebook (Ctrl+S)
2. Open the **Source Control** panel in VS Code (left sidebar, branch icon)
3. Stage your changes, write a commit message, and **Commit & Push**
4. Verify on GitHub that your notebook appears with all outputs visible
5. Submit your repository URL on **Canvas** (same URL as Part A)

> ⚠️ **Codespace warning:** Your Codespace may be deleted after a period of
> inactivity. Always push your work to GitHub — don't rely on the Codespace
> persisting.